# 01 — Exploratory Data Analysis

Project #24 — On-Device Quantized SLM Summarization. Inspect the 10k-doc corpus before evaluating summarizers. Run `python -m slm_quant.data` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROC = Path('../data/processed')
sns.set_theme(style='whitegrid')

In [ ]:
docs = pd.read_parquet(PROC / 'docs.parquet')
test = pd.read_parquet(PROC / 'test.parquet')
print(f'docs={len(docs):,}  test={len(test):,}')
docs.head()

## Schema and missingness

In [ ]:
print(docs.dtypes)
print('\nMissingness:')
print(docs.isna().sum())

## Domain mix

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
docs['domain'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Documents per domain')
plt.show()

## Document length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
sns.boxplot(data=docs, x='domain', y='doc_tokens', ax=ax)
ax.set_title('Document tokens per domain')
plt.show()

## Reference-summary length

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
sns.histplot(data=docs, x='summary_tokens', hue='domain', kde=True, ax=ax, bins=30)
ax.set_title('Reference-summary tokens by domain')
plt.show()

## Compression ratio (summary / doc)

In [ ]:
docs['compression'] = docs['summary_tokens'] / docs['doc_tokens']
fig, ax = plt.subplots(figsize=(8,3))
sns.violinplot(data=docs, x='domain', y='compression', ax=ax)
ax.set_title('Compression ratio per domain')
plt.show()

## Token-overlap heatmap (sanity)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(max_features=200, min_df=10)
X = vec.fit_transform(docs['doc_text'])
df = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
df['domain'] = docs['domain'].values
means = df.groupby('domain').mean()
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(means.T.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Per-domain mean token-frequency correlation')
plt.show()

## Target-leakage check
Reference summary is the first + last sentence of the document — by construction. Confirm reference is always a prefix-or-suffix substring of the document.

In [ ]:
leak_check = docs.apply(
    lambda r: r['ref_summary'].split()[0] in r['doc_text'].split()[:5],
    axis=1
).mean()
print(f'fraction where ref starts with the first doc sentence: {leak_check:.3f}')

## Takeaways
- Each domain has its own characteristic vocabulary; cross-domain correlation is moderate, not high.
- Reference summaries average ~12% of the document length — squarely in the small/medium budget tiers.